# Demo 1 — Critique and redesign

Begin with the question, audience, intended descriptive claim, displayed unit, grain, and variable roles before judging or repairing the chart.

Colab is the default launch path; the same source runs in clean local Jupyter. The setup cell installs only mismatched course packages before their first import. Colab files are ephemeral, and edits made in a GitHub-opened Colab tab are not automatically saved to GitHub.

This demo uses only course-authored synthetic prepared data. Do not add uploads, Drive mounts, network data, credentials, private records, or sensitive output. Stored notebook output is not execution evidence: restart and run all from a fresh runtime. Assignment Colab submission remains conditional on the repository-save/Classroom50 pilot.


## Start with the communication contract

- **Question:** How do the observed follow-up percentages differ by program and period?
- **Audience:** A clinic operations group deciding what deserves follow-up.
- **Intended claim:** The Reminder rows show a larger observed before-to-after increase in this prepared summary.
- **Unit and grain:** One row and one bar represent one program-period prepared percentage.
- **Variable roles:** Program and period are categorical; follow-up percentage is quantitative.

The table is descriptive. It cannot establish that a reminder program caused a change.


In [ ]:
import importlib.metadata
import platform
from pathlib import Path
import subprocess
import sys

EXPECTED_PACKAGES = {
    "numpy": "2.0.2",
    "pandas": "3.0.3",
    "matplotlib": "3.10.8",
    "seaborn": "0.13.2",
}

assert platform.python_version() == "3.12.13", (
    "Select the course Python 3.12.13 runtime before continuing; found "
    f"{platform.python_version()}."
)

install_specs = []
for package_name, expected_version in EXPECTED_PACKAGES.items():
    try:
        installed_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        install_specs.append(f"{package_name}=={expected_version}")

if install_specs:
    print("Installing mismatched course packages:", ", ".join(install_specs))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *install_specs]
    )

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

actual_versions = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "Matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
}
print(actual_versions)
assert actual_versions == {
    "Python": "3.12.13",
    "NumPy": "2.0.2",
    "pandas": "3.0.3",
    "Matplotlib": "3.10.8",
    "seaborn": "0.13.2",
}

BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
PURPLE = "#CC79A7"


In [ ]:
from hashlib import sha256

FIXTURES = {'followup_summary.csv': {'text': 'program,period,follow_up_percent\nStandard,Before,68\nStandard,After,69\nReminder,Before,67\nReminder,After,74\n', 'sha256': '928e929c0779800eb9f5b4cfbfadcafe0a0fc160d315265ce36182047c32c6f9'}}


def find_demo_directory(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "07" / "demo"):
            if (
                (candidate / "DEMO_GUIDE.md").is_file()
                and (candidate / ".python-version").is_file()
            ):
                return candidate
        if current.parent == current:
            return None
        current = current.parent


DEMO_DIRECTORY = find_demo_directory(Path.cwd())
if DEMO_DIRECTORY is None:
    DEMO_DIRECTORY = Path.cwd().resolve()

DATA_DIRECTORY = DEMO_DIRECTORY / "data"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIXTURE_PATHS = {}
for fixture_name, fixture_contract in FIXTURES.items():
    fixture_path = DATA_DIRECTORY / fixture_name
    if not fixture_path.exists():
        fixture_path.write_text(fixture_contract["text"], encoding="utf-8")
    actual_checksum = sha256(fixture_path.read_bytes()).hexdigest()
    assert actual_checksum == fixture_contract["sha256"], (
        f"{fixture_name} does not match the supplied fixture checksum. "
        "Restore the committed file; corrupt data are never replaced silently."
    )
    FIXTURE_PATHS[fixture_name] = fixture_path

OUTPUT_DIRECTORY = DEMO_DIRECTORY / "output"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
for output_name in ['followup_redesign.png']:
    output_path = OUTPUT_DIRECTORY / output_name
    if output_path.exists():
        output_path.unlink()

print("Demo directory:", DEMO_DIRECTORY)
print("Fixtures:", FIXTURE_PATHS)
print("Output directory:", OUTPUT_DIRECTORY)

followup = pd.read_csv(FIXTURE_PATHS["followup_summary.csv"])
expected_followup = pd.DataFrame(
    {
        "program": ["Standard", "Standard", "Reminder", "Reminder"],
        "period": ["Before", "After", "Before", "After"],
        "follow_up_percent": [68, 69, 67, 74],
    }
)
pd.testing.assert_frame_equal(followup, expected_followup)
print(followup)


## Diagnose the flawed chart

Before running the next cell, look for five defects: an unsupported causal title, a truncated magnitude baseline, a missing percentage unit, program identity encoded only by color, and distracting decoration. Name the defect and the comparison it distorts or obscures.


In [ ]:
periods = ["Before", "After"]
x_positions = np.arange(len(periods))
bar_width = 0.34

standard_values = (
    followup.loc[followup["program"].eq("Standard")]
    .set_index("period")
    .loc[periods, "follow_up_percent"]
    .to_numpy()
)
reminder_values = (
    followup.loc[followup["program"].eq("Reminder")]
    .set_index("period")
    .loc[periods, "follow_up_percent"]
    .to_numpy()
)

flawed_figure, flawed_ax = plt.subplots(figsize=(7, 4))
flawed_figure.patch.set_facecolor("#FFF4CC")
flawed_ax.bar(x_positions - bar_width / 2, standard_values, bar_width, label="Standard")
flawed_ax.bar(x_positions + bar_width / 2, reminder_values, bar_width, label="Reminder")
flawed_ax.set(
    title="Reminder program caused better follow-up",
    xlabel="Period",
    xticks=x_positions,
    xticklabels=periods,
    ylim=(66, 75),
)
flawed_ax.grid(True, axis="both", linewidth=1.5, color="#555555")
flawed_ax.legend()
flawed_figure.tight_layout()

assert flawed_ax.get_ylim()[0] > 0
assert flawed_ax.get_ylabel() == ""
assert "caused" in flawed_ax.get_title()
assert len(flawed_ax.patches) == 4


## Repair the comparison

The revision uses a zero baseline because bar length encodes magnitude, names the percentage unit and prepared-summary context, replaces the causal title with a descriptive one, removes decoration, and uses hatch as a redundant program cue in addition to color.


In [ ]:
corrected_figure, corrected_ax = plt.subplots(figsize=(7, 4))
standard_bars = corrected_ax.bar(
    x_positions - bar_width / 2,
    standard_values,
    bar_width,
    color=BLUE,
    edgecolor="#222222",
    hatch="//",
    label="Standard",
)
reminder_bars = corrected_ax.bar(
    x_positions + bar_width / 2,
    reminder_values,
    bar_width,
    color=ORANGE,
    edgecolor="#222222",
    hatch="\\",
    label="Reminder",
)
corrected_ax.set(
    title="Observed follow-up percentages in the prepared summary",
    xlabel="Observation period",
    ylabel="Follow-up (%)",
    xticks=x_positions,
    xticklabels=periods,
    ylim=(0, 80),
)
corrected_ax.bar_label(standard_bars, fmt="%d%%", padding=2)
corrected_ax.bar_label(reminder_bars, fmt="%d%%", padding=2)
corrected_ax.legend(
    title="Program",
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
)
corrected_ax.spines[["top", "right"]].set_visible(False)
corrected_figure.tight_layout()

followup_text_alternative = (
    "Grouped bar chart of prepared follow-up percentage by observation period "
    "and program. Standard rises from 68% to 69%, while Reminder rises from "
    "67% to 74%. These descriptive prepared summaries do not establish that "
    "the reminder program caused the difference."
)

corrected_path = OUTPUT_DIRECTORY / "followup_redesign.png"
corrected_figure.savefig(corrected_path, dpi=150, bbox_inches="tight")

assert corrected_ax.get_ylim()[0] == 0
assert corrected_ax.get_ylabel() == "Follow-up (%)"
assert "caused" not in corrected_ax.get_title().lower()
assert len(corrected_ax.patches) == 4
assert {patch.get_hatch() for patch in corrected_ax.patches} == {"//", "\\"}
assert corrected_ax.get_legend() is not None
assert "do not establish" in followup_text_alternative
assert corrected_path.is_file() and corrected_path.stat().st_size > 1_000
corrected_pixels = plt.imread(corrected_path)
assert corrected_pixels.shape[0] > 300 and corrected_pixels.shape[1] > 500

print(followup_text_alternative)
print("Wrote:", corrected_path)


## Human visual QA

Automated checks cannot certify communicative quality. Inspect the newly rendered chart: are the bar lengths comparable from zero, labels and percentages readable, hatch and color both usable, decoration subordinate to the data, title descriptive, legend close to the marks, and text alternative accurate about both the pattern and limitation?


In [ ]:
assert corrected_path.exists()
assert standard_values.tolist() == [68, 69]
assert reminder_values.tolist() == [67, 74]
print("Demo 1 fresh-run verification passed")
